# Human WAT snRNA-seq: the PRM1+ "unknown" cluster and visceral vs subcutaneous adipocytes

Dataset: *Single-Nucleus Analysis of Human White Adipose Tissue Reveals Adipocyte Subsets with Distinct Metabolic Profiles* (CELLxGENE collection `6b701826-37bb-4356-9792-ff41fc4c3161`).

Set `WAT_DATA_DIR` to control where the ~1 GB H5AD is stored (default `~/adipose_cxg/data`). Existing files are validated, not re-downloaded.

In [1]:
import os, sys, warnings
os.environ.setdefault("NUMBA_CACHE_DIR", "/tmp/numba_cache"); warnings.filterwarnings("ignore")
sys.path.insert(0, os.path.join(os.getcwd(), "..", "src")); sys.path.insert(0, "src")
import numpy as np, pandas as pd, scanpy as sc, anndata as ad, matplotlib as mpl, matplotlib.pyplot as plt
from scipy.stats import gaussian_kde, spearmanr, norm
from statsmodels.stats.multitest import multipletests
from adjustText import adjust_text
import wat_pipeline as wp

ROOT = os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
FIG, RES = os.path.join(ROOT, "figures"), os.path.join(ROOT, "results")
os.makedirs(FIG, exist_ok=True); os.makedirs(RES, exist_ok=True)
DATA = wp.data_dir(); ORIG = DATA / "wat_snrna_original.h5ad"; ADIP = DATA / "wat_snrna_adipocytes.h5ad"
mpl.rcParams.update({"font.size": 8, "axes.titlesize": 8, "axes.labelsize": 8, "legend.fontsize": 7, "xtick.labelsize": 6, "ytick.labelsize": 6})
sc.settings.verbosity = 0

## 1. Download or validate

In [3]:
a = wp.download_or_validate(ORIG).to_memory()
wp.gene_symbols(a)
print(a.shape, "| X log-normalised, raw.X integer UMI |", a.obs.donor_id.nunique(), "donors,", a.obs.library_id.nunique(), "libraries")
a.obs.cell_type.value_counts()

(65668, 35476) | X log-normalised, raw.X integer UMI | 19 donors, 22 libraries


cell_type
preadipocyte                            15744
adipocyte                               15205
macrophage                               7582
endothelial cell of venule               5526
mesothelial cell                         4060
unknown                                  3007
neuron associated cell                   2917
endothelial stalk cell                   2495
endothelial cell of arteriole            2069
smooth muscle cell                       1896
pericyte                                 1702
endothelial cell of lymphatic vessel     1513
T cell                                   1504
mast cell                                 187
B cell                                    142
Schwann cell                               96
erythroblast                               23
Name: count, dtype: int64

## 2. Overview: *ADIPOQ* and cell type on the authors' UMAP

In [5]:
um = a.obsm["X_umap"]; ct = a.obs.cell_type.astype(str); rng = np.random.default_rng(0)
expr = a[:, "ADIPOQ"].X.toarray().ravel(); o = np.argsort(expr); perm = rng.permutation(a.n_obs)
cats = ct.value_counts().index.tolist(); pal = dict(zip(cats, plt.cm.tab20(np.linspace(0, 1, len(cats)))))
short = {"endothelial cell of venule": "venular EC", "endothelial cell of arteriole": "arteriolar EC", "endothelial stalk cell": "stalk EC",
         "endothelial cell of lymphatic vessel": "lymphatic EC", "neuron associated cell": "neuron-assoc.", "smooth muscle cell": "smooth muscle"}
fig, axes = plt.subplots(1, 2, figsize=(11, 5.2))
s = axes[0].scatter(um[o, 0], um[o, 1], c=expr[o], s=1.5, cmap="viridis", linewidths=0, rasterized=True)
fig.colorbar(s, ax=axes[0], shrink=0.6, pad=0.01).set_label("$ADIPOQ$ log-normalised expression")
axes[0].set_title("$ADIPOQ$ marks the adipocyte island", loc="left")
axes[1].scatter(um[perm, 0], um[perm, 1], c=[pal[c] for c in ct.values[perm]], s=1.5, linewidths=0, rasterized=True)
axes[1].set_title("CELLxGENE cell-type annotation", loc="left")
for c in cats:
    idx = np.where(ct.values == c)[0]
    if len(idx) < 1000: continue
    sub = um[rng.choice(idx, min(2000, len(idx)), replace=False)]; x, y = sub[np.argmax(gaussian_kde(sub.T)(sub.T))]
    axes[1].text(x, y, short.get(c, c), fontsize=6, ha="center", va="center", bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="none", alpha=0.8))
axes[1].legend(handles=[mpl.lines.Line2D([], [], marker="o", ls="", ms=4, color=pal[c], label=c) for c in cats if (ct.values == c).sum() < 1000],
               loc="upper left", frameon=False, title="rare types")
for ax in axes:
    ax.set_xticks([]); ax.set_yticks([]); ax.set_xlabel("UMAP 1"); ax.set_ylabel("UMAP 2")
    for sp in ax.spines.values(): sp.set_visible(False)
fig.tight_layout(); fig.savefig(os.path.join(FIG, "umap_adipoq_celltype.png"), dpi=200)

## 3. What is the "unknown" cluster?

Donor-stratified t-test of `unknown` against every other cell type (per donor with ≥20 cells of both, Stouffer-combined, weights √n). Depot/T2D/sex are constant within donor, so stratifying by donor removes them.

In [7]:
det_all = np.asarray((a.X > 0).mean(0)).ravel(); det_unk = np.asarray((a[a.obs.cell_type == "unknown"].X > 0).mean(0)).ravel()
ad_ = a[:, (det_all > 0.01) | (det_unk > 0.05)].copy()
res = {}
for other in [c for c in cats if c != "unknown"]:
    r = wp.stratified_ttest(ad_, "cell_type", "unknown", other, "donor_id")
    if len(r): res[other] = r.assign(comparison=f"unknown_vs_{other}")
full = pd.concat(res.values()); full.index.name = "gene"
full.to_csv(os.path.join(RES, "unknown_vs_celltypes_donorstrat_ttest.csv.gz"))
piv = full.reset_index().pivot(index="gene", columns="comparison", values="stouffer_z")
consensus_up = piv.min(1).sort_values(ascending=False)
consensus_up.head(20).round(0).to_frame("min Stouffer z across all comparisons")

,min Stouffer z across all comparisons
gene,
ODF2,86.0
PRM1,86.0
TUBA3C,83.0
ENSG00000187186,81.0
TNP1,78.0
OAZ3,77.0
UBB,77.0
PGP,76.0
DYNLT2,72.0


In [8]:
genes = ["PRM1", "TNP1", "ODF2", "TUBA3C", "OAZ3", "GAPDHS", "FABP9", "MEIG1", "GPX4", "UBB", "ADIPOQ", "PDGFRA", "MRC1", "MSLN", "PTPRC"]
ad_.obs["cell_type_short"] = ad_.obs.cell_type.astype(str).map(lambda c: short.get(c, c))
dp = sc.pl.dotplot(ad_, genes, groupby="cell_type_short", use_raw=False, standard_scale="var", return_fig=True, figsize=(8.5, 5),
                   title="The 'unknown' cluster is a spermatid transcript signature")
dp.savefig(os.path.join(FIG, "unknown_spermatid_dotplot.png"), dpi=200)
a.obs.groupby("donor_id", observed=True).agg(sex=("sex", "first"), pct_unknown=("cell_type", lambda s: round((s == "unknown").mean() * 100, 1))).sort_values("pct_unknown", ascending=False).head(9)

,sex,pct_unknown
donor_id,,
donor-GOLD,female,16.1
donor-BOAT,female,16.0
donor-KISS,female,8.8
donor-PASS,male,6.9
donor-TREE,female,6.5
donor-KEYS,female,4.2
donor-APEX,male,3.6
donor-DOWN,female,3.4
donor-CORK,male,2.9


**Interpretation.** Protamine/transition-protein and spermiogenesis genes (*PRM1*, *TNP1*, *ODF2*, *TUBA3C*, *OAZ3*, *GAPDHS*, *FABP9*, *MEIG1*) are detected in >80% of these nuclei and essentially nowhere else, consistently across donors — most of them female. This is a technical contaminant (index hopping / cross-contamination from a testis library), not an adipose cell type. It is excluded from everything below.

## 4. Adipocyte-only object (raw counts + depot / donor / library)

In [11]:
adi = wp.build_adipocyte_object(a, ADIP)
del a, ad_
print(adi.shape, adi.obs.depot.value_counts().to_dict(), "| donors:", adi.obs.donor.nunique(), "| libraries:", adi.obs.library.nunique())
lib = adi.obs.groupby("library", observed=True).agg(depot=("depot", "first"), donor=("donor", "first"), n=("depot", "size"))
paired = lib.donor.value_counts().pipe(lambda s: s[s > 1].index.tolist())
print("libraries per depot:", lib.depot.value_counts().to_dict(), "| donors with both depots:", paired)

(15205, 35476) {'visceral': 10584, 'subcutaneous': 4621} | donors: 19 | libraries: 22
libraries per depot: {'visceral': 12, 'subcutaneous': 10} | donors with both depots: ['TREE', 'GOLD', 'BLUE']


## 5. Visceral vs subcutaneous adipocytes

Depot is nested in donor (only 3 donors contribute both depots), so the **primary analysis is pseudobulk per library** (DESeq2, `~ depot`; each library = one donor-depot). For comparison, a **cell-level donor-stratified t-test** is run within the 3 paired donors (the only cells where donor can be controlled at cell level), plus a naive cell-level t-test.

In [13]:
counts, meta = wp.pseudobulk(adi, "library")
pb = wp.run_deseq2(counts, meta)
print("pseudobulk genes tested:", len(pb), "| padj<0.05:", (pb.pb_padj < 0.05).sum(), "| up in visceral:", ((pb.pb_padj < 0.05) & (pb.pb_log2FC > 0)).sum())

pseudobulk genes tested: 27217 | padj<0.05: 3072 | up in visceral: 937


In [14]:
adc = adi.copy(); sc.pp.normalize_total(adc, target_sum=1e4); sc.pp.log1p(adc); adc = adc[:, pb.index].copy()
sc.tl.rank_genes_groups(adc, "depot", groups=["visceral"], reference="subcutaneous", method="t-test", n_genes=adc.n_vars, use_raw=False)
naive = sc.get.rank_genes_groups_df(adc, "visceral").set_index("names")[["scores", "pvals", "pvals_adj"]]
naive.columns = ["cell_naive_t", "cell_naive_pval", "cell_naive_padj"]
strat = wp.stratified_ttest(adc[adc.obs.donor.isin(paired)].copy(), "depot", "visceral", "subcutaneous", "donor")
strat = strat[["stouffer_z", "pval", "padj", "sign_consistency"]].add_prefix("cell_donorstrat_")
V = adc[adc.obs.depot == "visceral"].X; S = adc[adc.obs.depot == "subcutaneous"].X
desc = pd.DataFrame({"pct_visceral": np.asarray((V > 0).mean(0)).ravel(), "pct_subcutaneous": np.asarray((S > 0).mean(0)).ravel(),
                     "cell_log2FC": np.log2((np.asarray(np.expm1(V).mean(0)).ravel() + 1e-2) / (np.asarray(np.expm1(S).mean(0)).ravel() + 1e-2))}, index=adc.var_names)
tab = pb.join(desc).join(naive).join(strat).sort_values("pb_pval"); tab.index.name = "gene"
tab.to_csv(os.path.join(RES, "adipocyte_visceral_vs_subcutaneous_DE.csv"))
sig = tab[tab.pb_padj < 0.05]
print("Spearman(pseudobulk stat, naive t) =", round(spearmanr(tab.pb_stat, tab.cell_naive_t, nan_policy="omit")[0], 2),
      "| Spearman(pseudobulk stat, donor-strat z) =", round(spearmanr(tab.pb_stat, tab.cell_donorstrat_stouffer_z, nan_policy="omit")[0], 2))
print("sign agreement on pseudobulk-significant genes:", round((np.sign(sig.pb_stat) == np.sign(sig.cell_donorstrat_stouffer_z)).mean(), 2))
print("significant genes — pseudobulk:", len(sig), "| naive cell-level:", (tab.cell_naive_padj < 0.05).sum(), "| donor-stratified cell-level:", (tab.cell_donorstrat_padj < 0.05).sum())
tab.head(20)[["pb_log2FC", "pb_padj", "cell_log2FC", "cell_donorstrat_stouffer_z", "pct_visceral", "pct_subcutaneous"]].round(3)

Spearman(pseudobulk stat, naive t) = 0.92 | Spearman(pseudobulk stat, donor-strat z) = 0.5
sign agreement on pseudobulk-significant genes: 0.86
significant genes — pseudobulk: 3072 | naive cell-level: 11817 | donor-stratified cell-level: 8193


,pb_log2FC,pb_padj,cell_log2FC,cell_donorstrat_stouffer_z,pct_visceral,pct_subcutaneous
gene,,,,,,
CRNDE,-4.963,0.0,-4.058,-14.995,0.012,0.296
HOXC6,-5.502,0.0,-4.127,-14.474,0.004,0.188
NRCAM,-6.057,0.0,-4.973,-17.660,0.015,0.357
HOXC8,-6.344,0.0,-3.797,-11.943,0.001,0.110
SIM1,-6.221,0.0,-4.706,-7.743,0.003,0.157
HOXC9,-5.898,0.0,-3.045,-8.932,0.001,0.079
LINC02381,-2.440,0.0,-1.848,-7.269,0.022,0.110
HOXC4,-4.199,0.0,-3.037,-10.868,0.007,0.113
COL11A1,-4.781,0.0,-3.519,-7.345,0.008,0.114


## 6. Volcano plots

In [16]:
t = tab.dropna(subset=["pb_padj"]).copy()
t["pb_nlp"] = -np.log10(t.pb_padj.clip(1e-300)); t["cs_nlp"] = -np.log10(t.cell_donorstrat_padj.clip(1e-300))
t["cls"] = np.where(t.pb_padj >= 0.05, "ns", np.where(t.pb_log2FC > 0, "vis", "sq"))
col = {"ns": "0.8", "vis": "#b2182b", "sq": "#2166ac"}; lbl = {"ns": "not significant (pseudobulk)", "sq": "higher in subcutaneous", "vis": "higher in visceral"}
sig = t[t.cls != "ns"]; lab = [g for g in list(sig.sort_values("pb_stat", ascending=False).head(12).index) + list(sig.sort_values("pb_stat").head(12).index) if not g.startswith("ENSG")]
spec = [("pb_log2FC", "pb_nlp", "log$_2$ fold change (visceral / subcutaneous), pseudobulk", "−log$_{10}$ BH-adjusted p (DESeq2)", "Pseudobulk per library: subcutaneous adipocytes keep a HOXC/TBX15 positional identity"),
        ("cell_log2FC", "cs_nlp", "log$_2$ fold change (visceral / subcutaneous), cell means", "−log$_{10}$ BH-adjusted p (donor-stratified t-test, 3 paired donors)", "Cell-level test within paired donors: same top genes, inflated significance")]
fig, axes = plt.subplots(1, 2, figsize=(11, 5.2))
for ax, (x, y, xl, yl, ttl) in zip(axes, spec):
    for c in ["ns", "sq", "vis"]:
        s = t[t.cls == c]; ax.scatter(s[x], s[y], s=4 if c == "ns" else 7, c=col[c], linewidths=0, rasterized=True, label=lbl[c])
    texts = [ax.text(t.loc[g, x], t.loc[g, y], g, fontsize=6, style="italic", color=col[t.loc[g, "cls"]]) for g in lab]
    adjust_text(texts, ax=ax, arrowprops=dict(arrowstyle="-", color="0.4", lw=0.4), expand=(1.3, 1.5))
    ax.set_xlabel(xl); ax.set_ylabel(yl); ax.set_title(ttl, loc="left"); ax.margins(0.04)
    for sp in ["top", "right"]: ax.spines[sp].set_visible(False)
axes[0].axhline(-np.log10(0.05), color="0.5", lw=0.6, ls=":"); axes[0].legend(frameon=False, loc="upper right", markerscale=2)
axes[1].set_ylim(0, 310); axes[1].text(0.98, 0.98, "y capped at 300", transform=axes[1].transAxes, ha="right", va="top", fontsize=6, color="0.4")
fig.tight_layout(); fig.savefig(os.path.join(FIG, "adipocyte_depot_volcano.png"), dpi=200)

**Interpretation.** Subcutaneous adipocytes retain a developmental positional code (*HOXC4/6/8/9*, *TBX15*, *TWIST1*, *PAX3*, *SIM1*, *NRCAM*, *COL11A1*); visceral adipocytes show a weaker, diffuse shift (*ENOX1*, *MAPK10*, *CUBN*, *MGMT*, *PRKAR2B*, *PIK3CA*). The cell-level test recovers the same genes but calls several-fold more genes significant — pseudo-replication — so the pseudobulk table is the one to cite.